<a href="https://colab.research.google.com/github/kmmmm25/KumaGPT/blob/main/Kuma_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [64]:
class Tokenizer:
    def __init__(self, chars: list[str]) -> None:
        self.str_to_idx: dict[str, int] = dict()
        self.str_to_idx['<|endoftext|>'] = 0
        # utf-8
        for i in range(256):
            if f'<utf8_{i}>' not in self.str_to_idx:
                self.str_to_idx[f'<utf8_{i}>'] = len(self.str_to_idx)
        for char in chars:
            self.str_to_idx[char] = len(self.str_to_idx) if char not in self.str_to_idx else self.str_to_idx[char]

        # 登録したIDに重複がないか確認
        assert len(self.str_to_idx.values()) == len(set(self.str_to_idx.values()))

        self.idx_to_str: dict[int, str] = dict()
        for key, value in self.str_to_idx.items():
            self.idx_to_str[value] = key

    def encode(self, text: str, eot=False) -> list[int]:
        result: list[int] = []
        for char in text:
            maybe_token: int | None = self.str_to_idx.get(char)
            if maybe_token is not None:
                result.append(self.str_to_idx[char])
            else:
                utf_8_num: list[int] = list(char.encode("utf-8"))
                for num in utf_8_num:
                    result.append(self.str_to_idx[f'<utf8_{num}>'])
        if eot:
            result.append(self.str_to_idx['<|endoftext|>'])
        return result

    def decode(self, tokens: list[int]) -> str:
        decoded_with_utf_token: list[str] = [self.idx_to_str[token] for token in tokens]
        decoded_postprocess_utf: list[str] = []
        utf_tokens: list[int] = []
        for token in decoded_with_utf_token:
            if token.startswith("<utf8_"):
                utf_num = int(token.replace("<utf8_", "").replace(">", ""))
                utf_tokens.append(utf_num)
            else:
                if utf_tokens:
                    decoded_postprocess_utf.append(bytes(utf_tokens).decode("utf-8", errors="replace"))
                    utf_tokens = []
                decoded_postprocess_utf.append(token)
        if utf_tokens:
            decoded_postprocess_utf.append(bytes(utf_tokens).decode("utf-8", errors="replace"))
            utf_tokens = []
        return "".join(decoded_postprocess_utf)

    def decode_with_utf(self, tokens:list[int]) -> str:
        return "".join([self.idx_to_str[token] for token in tokens])

In [65]:
!pip install -q datasets

In [66]:
from datasets import load_dataset

dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.ja",
    split="train",
    streaming=True
)

dataset = dataset.shuffle(
    seed=42,
    buffer_size=10_000
)

In [69]:
train_texts = []
val_texts = []
count = 0

# Iterate through the first dataset (IVUL-KAUST/MOLE)
for item in dataset:
  if count < 45000:
    if "text" in item:
      train_texts.append(item["text"])
  elif count < 50000:
    if "text" in item:
        val_texts.append(item["text"])
  else:
    break

  count += 1

all_text = "\n".join(train_texts)

In [70]:
all_text = "".join(train_texts)

vocab = sorted(set(all_text))

tokenizer = Tokenizer(vocab)

vocab_size = len(tokenizer.str_to_idx)

print(vocab_size)

9393


In [71]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(2005)

class Attention(nn.Module):
  def __init__(self, d_model, d_head, h):
    assert d_head % h == 0

    super().__init__()

    self.Wk = nn.Linear(d_model, d_head)
    self.Wq = nn.Linear(d_model, d_head)
    self.Wv = nn.Linear(d_model, d_head)

    self.h = h

    self.linear = nn.Linear(d_head, d_model)

  def forward(self, x):

    k = self.Wk(x)
    q = self.Wq(x)
    v = self.Wv(x)

    B, T, d_head = k.shape

    k = k.view(B, T, self.h, d_head // self.h).transpose(1, 2)
    q = q.view(B, T, self.h, d_head // self.h).transpose(1, 2)
    v = v.view(B, T, self.h, d_head // self.h).transpose(1, 2)

    y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    # (B, h, T, d_head)
    #          ↓
    # (B, T, h, d_head)
    y = y.transpose(1, 2)

    attention = y.reshape(B, T, d_head)

    output = self.linear(attention)

    return output

class Block(nn.Module):
  def __init__(self, d_model, d_head, d_ff, h):
    super().__init__()
    self.ln1 = nn.LayerNorm(d_model)
    self.attn = Attention(d_model, d_head, h)

    self.ln2 = nn.LayerNorm(d_model)
    self.ff = nn.Sequential(
        nn.Linear(d_model, d_ff),
        nn.GELU(),
        nn.Linear(d_ff, d_model)
    )

    self.dropout = nn.Dropout(0.1)

  def forward(self, x):
    x = x + self.attn(self.ln1(x))
    x = self.dropout(x)

    x = x + self.ff(self.ln2(x))
    x = self.dropout(x)

    return x


class Kuma_GPT(nn.Module):
  def __init__(self, vocab_size, block_size, d_model, d_head, d_ff, n_layer, h):
    super().__init__()
    self.token_embedding = nn.Embedding(vocab_size, d_model)
    self.pos_embedding = nn.Embedding(block_size, d_model)

    self.blocks = nn.ModuleList([
        Block(d_model, d_head, d_ff, h)
        for _ in range(n_layer)
    ])

    self.lnf = nn.LayerNorm(d_model)
    self.linear_output = nn.Linear(d_model, vocab_size)

    self.block_size = block_size

  def forward(self, input, targets=None):
    B, T = input.shape

    tok_emb = self.token_embedding(input)

    pos_input = torch.arange(T, device=input.device)
    pos_emb = self.pos_embedding(pos_input)

    x = tok_emb + pos_emb

    for block in self.blocks:
      x = block(x)

    y = self.lnf(x)
    logits = self.linear_output(y)

    loss = None
    if targets is not None:
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-1)

    return logits, loss

  @torch.no_grad()
  def generate(self, input, max, temperature=0.7, top_k=30):
    self.eval()
    results = input.clone()

    for _ in range(max):
        x = results[:, -self.block_size:]
        logits, _ = self(x)

        next_logits = logits[:, -1, :] / temperature

        if top_k is not None:
            values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
            next_logits[next_logits < values[:, [-1]]] = -float("inf")

        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        results = torch.cat([results, next_token], dim=1)

    return results

In [72]:
block = 256
d_model = 256
d_ff = d_model * 4
d_head = d_model
n_layer = 6
h = 4

In [73]:
train = train_texts[:]

print(len(train_texts))

train_tokens = []
for x in train:
  tokens = tokenizer.encode(x)
  train_tokens.append(tokens)


val = val_texts[:]

print(len(val_texts))

val_tokens = []
for x in val:
  tokens = tokenizer.encode(x)
  val_tokens.append(tokens)

45000
5000


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_flat_tokens = [token for sublist in train_tokens for token in sublist]

batch_size = 32

chunk_size = block + 1
num_chunks = len(train_flat_tokens) // chunk_size
train_tokens = torch.tensor(train_flat_tokens[:num_chunks * chunk_size]).view(num_chunks, chunk_size)

val_flat_tokens = [token for sublist in val_tokens for token in sublist]

chunk_size = block + 1
num_chunks = len(val_flat_tokens) // chunk_size
val_tokens = torch.tensor(val_flat_tokens[:num_chunks * chunk_size]).view(num_chunks, chunk_size)

epoch = 4

model = Kuma_GPT(vocab_size, block, d_model, d_head, d_ff, n_layer, h).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
    eta_min=1e-5
)

for i in range(epoch):
    #train
    model.train()

    total_train_loss = 0
    num_batches = 0
    for j in range(0, len(train_tokens), batch_size):

      batch_token = train_tokens[j:j+batch_size].to(device)

      input = batch_token[:, :-1]
      target = batch_token[:, 1:]

      optimizer.zero_grad()

      logits, loss = model(input, target)

      loss.backward()
      optimizer.step()

      total_train_loss += loss.item()
      num_batches += 1

    scheduler.step()

    train_loss = total_train_loss / num_batches

    #val
    model.eval()

    total_val_loss = 0
    num_batches = 0

    with torch.no_grad():
      for j in range(0, len(val_tokens), batch_size):

        batch_token = val_tokens[j:j+batch_size].to(device)

        input = batch_token[:, :-1]
        target = batch_token[:, 1:]

        logits, loss = model(input, target)

        total_val_loss += loss.item()
        num_batches += 1

    val_loss = total_val_loss / num_batches




    print(f'epoch {i + 1}')

    print(f'train loss: {train_loss:.6f}')
    print(f'val loss: {val_loss:.6f}')

    print('')

In [11]:
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Batch size: {batch_size}")
print(f"Block size: {block}")
print(f"Tokens / step: {batch_size * block:,}")
print(f"Training steps: {num_chunks:,}")
print(f"Total training tokens: {batch_size * block * num_chunks:,}")
print(f"Total training tokens: {batch_size * block * num_chunks / 1e6:.2f}M")
print("")
print(f'val loss: {val_loss:.6f}')
print(f'train loss: {train_loss:.6f}')

Parameters: 7,651,758
Batch size: 32
Block size: 256
Tokens / step: 8,192
Training steps: 2,710
Total training tokens: 22,200,320
Total training tokens: 22.20M

val loss: 2.974706
train loss: 2.589438


In [20]:
sentence = "今回の実験"
sentence_token = tokenizer.encode(sentence)

x = torch.tensor(sentence_token, dtype=torch.long).unsqueeze(0).to(device)

y = model.generate(x, 50)

print(tokenizer.decode(y[0].tolist()))

今回の実験を取っていた。その際には、当時の国際連合安全保障理事会決議6590、UNANCAA Pomerite


In [21]:
print(model)
print(next(model.parameters()).device)

Kuma_GPT(
  (token_embedding): Embedding(5550, 256)
  (pos_embedding): Embedding(256, 256)
  (blocks): ModuleList(
    (0-5): 6 x Block(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): Attention(
        (Wk): Linear(in_features=256, out_features=256, bias=True)
        (Wq): Linear(in_features=256, out_features=256, bias=True)
        (Wv): Linear(in_features=256, out_features=256, bias=True)
        (linear): Linear(in_features=256, out_features=256, bias=True)
      )
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1024, out_features=256, bias=True)
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (lnf): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (linear_output): Linear(in_features=256, out_features=5550, bias=True)
)
cuda:0


In [22]:
from google.colab import drive
import os

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/KumaGPT/checkpoints"
os.makedirs(save_dir, exist_ok=True)

Mounted at /content/drive


In [23]:
config = {
    "vocab_size": vocab_size,
    "block_size": block,
    "d_model": d_model,
    "d_head": d_head,
    "d_ff": d_ff,
    "n_layer": n_layer,
    "h": h,
}

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),

    "epoch": epoch,
    "train_loss": train_loss,
    "val_loss": val_loss,

    "config": config,
    "tokenizer_str_to_idx": tokenizer.str_to_idx,
}

save_path = f"{save_dir}/pretrained_base.pt"

torch.save(checkpoint, save_path)

print(f"保存完了: {save_path}")

保存完了: /content/drive/MyDrive/KumaGPT/checkpoints/pretrained_base.pt


In [24]:
print(os.path.exists(save_path))
print(f"{os.path.getsize(save_path) / 1024**2:.2f} MB")

True
87.77 MB


In [25]:
loaded_checkpoint = torch.load(
    save_path,
    map_location=device,
    weights_only=False
)

print(loaded_checkpoint.keys())
print("保存epoch:", loaded_checkpoint["epoch"])
print("validation loss:", loaded_checkpoint["val_loss"])

dict_keys(['model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'epoch', 'train_loss', 'val_loss', 'config', 'tokenizer_str_to_idx'])
保存epoch: 50
validation loss: 2.9747064927045037


# Full Fine Tuning

In [26]:
from datasets import load_dataset

dataset = load_dataset('llm-jp/oasst1-21k-ja', split='train')

print(dataset['conversations'][4][0])
print(dataset['conversations'][4][0]['from'])

{'from': 'human', 'value': '「速く走るためにゆっくり走る」という言葉を、長距離走と持久力トレーニングの文脈で説明してください。また、科学的根拠の参考文献も添えてください。'}
human


In [27]:
def apply_chat_template(example):
  formatted_messages = []
  for message in example['conversations']:
    print(example)
    role = message['from']
    value = message['value'].strip()

    if role == "human":
      formatted_messages.append(f"ユーザー：{value}")

    elif role == "gpt":
      formatted_messages.append(f"アシスタント：{value}<|endoftext|>")

  text = "\n".join(formatted_messages)

  return {"text": text}

In [28]:
formatted_dataset = dataset.map(
    apply_chat_template,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/21164 [00:00<?, ? examples/s]

ストリーミング出力は最後の 5000 行に切り捨てられました。
{'conversations': [{'from': 'human', 'value': '点の列を受け取り、それらすべてを通る最短経路を返すPythonプログラムを書きなさい。'}, {'from': 'gpt', 'value': '「巡回セールスマン問題」のアルゴリズムを使って、点の列を受け取り、それらすべてを通る最短経路を返すPythonプログラムの例である：\n\n```\nfrom itertools import permutations\n\ndef tsp(points)：\n    # 全ての点間の距離を計算\n    def distance(p1, p2)：\n        x1, y1 = p1\n        x2, y2 = p2\n        return ((x1 - x2) ** 2 + (y1 - y2) ** 2) ** 0.5\n\n    # 点のすべての可能な順列を得る\n    all_paths = permutations(points)\n\n    # 各パスの距離を計算し、最短のパスを記録する\n    shortest_path = float("inf")\n    shortest_path_points = [].\n    for path in all_paths：\n        total_distance = 0\n        for i in range(len(path) - 1)：\n            total_distance += distance(path[i], path[i + 1])\n        if total_distance < shortest_path：\n            shortest_path = total_distance\n            shortest_path_points = パス\n\n    return shortest_path, shortest_path_points\n\npoints = [(0, 0), (1, 2), (3, 4), (5, 6)].\nshortest_path, path = tsp(points

In [36]:
print(formatted_dataset)
print(formatted_dataset[1]['text'])
print(len(formatted_dataset['text']))

Dataset({
    features: ['text'],
    num_rows: 21164
})
ユーザー：スリーボディとは？
アシスタント：もっと詳しく説明してください。<|endoftext|>
ユーザー：物理学
アシスタント：物理学における3体問題とは、3つの点質量の初期位置と速度がわかっているときに、その後の位置と速度を求める問題である。三体問題のすべての事例を解くことのできる閉形式は存在しないので、数値的手法によって解かれることが多い。<|endoftext|>
21164


In [39]:
# 21164 x 0.9 = 19047.6

train = formatted_dataset[:19047]['text']

print(len(train))

train_tokens = []
for x in train:
  tokens = tokenizer.encode(x)
  train_tokens.append(tokens)

val = formatted_dataset[19047:]['text']

print(len(val))

val_tokens = []
for x in val:
  tokens = tokenizer.encode(x)
  val_tokens.append(tokens)

19047
2117


# 特殊トークン確認

In [35]:
end_text = "<|endoftext|>"
end_id = tokenizer.str_to_idx[end_text]

test_tokens = tokenizer.encode(end_text)

print("特殊トークンID:", end_id)
print("encode結果:", test_tokens)
print("トークン数:", len(test_tokens))

特殊トークンID: 0
encode結果: [287, 351, 328, 337, 327, 338, 329, 343, 328, 347, 343, 351, 289]
トークン数: 13


# "<|endoftext|>"が特殊トークンとして認識されていないので、Tokenizerのencode部分を直し、Transformerのgeneratorもそれに伴って修正

# sft lr scheduler　必須

In [50]:
train_flat_tokens = [token for sublist in train_tokens for token in sublist]
train_tokens = torch.tensor(train_flat_tokens[:num_chunks * chunk_size]).view(num_chunks, chunk_size)

val_flat_tokens = [token for sublist in val_tokens for token in sublist]
val_tokens = torch.tensor(val_flat_tokens[:num_chunks * chunk_size]).view(num_chunks, chunk_size)

epoch = 15

sft_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for i in range(epoch):
    #train
    model.train()

    total_train_loss = 0
    num_batches = 0
    for j in range(0, len(train_tokens), batch_size):

      batch_token = train_tokens[j:j+batch_size].to(device)

      input = batch_token[:, :-1]
      target = batch_token[:, 1:]

      sft_optimizer.zero_grad()

      logits, loss = model(input, target)

      loss.backward()
      sft_optimizer.step()

      total_train_loss += loss.item()
      num_batches += 1

    scheduler.step()

    train_loss = total_train_loss / num_batches

    #val
    model.eval()

    total_val_loss = 0
    num_batches = 0

    with torch.no_grad():
      for j in range(0, len(val_tokens), batch_size):

        batch_token = val_tokens[j:j+batch_size].to(device)

        input = batch_token[:, :-1]
        target = batch_token[:, 1:]

        logits, loss = model(input, target)

        total_val_loss += loss.item()
        num_batches += 1

    val_loss = total_val_loss / num_batches




    print(f'epoch {i + 1}')

    print(f'train loss: {train_loss:.6f}')
    print(f'val loss: {val_loss:.6f}')

    print('')

epoch 1
train loss: 2.376182
val loss: 2.451735

epoch 2
train loss: 2.331678
val loss: 2.444878

epoch 3
train loss: 2.300865
val loss: 2.434551

epoch 4
train loss: 2.272274
val loss: 2.429913

epoch 5
train loss: 2.246547
val loss: 2.426181

epoch 6
train loss: 2.223487
val loss: 2.421900

epoch 7
train loss: 2.202806
val loss: 2.422208

epoch 8
train loss: 2.182043
val loss: 2.422283

epoch 9
train loss: 2.160348
val loss: 2.420764

epoch 10
train loss: 2.140197
val loss: 2.422524

epoch 11
train loss: 2.121222
val loss: 2.420399

epoch 12
train loss: 2.104878
val loss: 2.418741

epoch 13
train loss: 2.089255
val loss: 2.420380

epoch 14
train loss: 2.073964
val loss: 2.419918

epoch 15
train loss: 2.058072
val loss: 2.425004



In [51]:
user = 'ユーザー：'
assistant = 'アシスタント:'

In [57]:
kaiwa = '日本の首都はどこですか？'
sentence = user + kaiwa + '\n' + assistant
sentence_token = tokenizer.encode(sentence)

x = torch.tensor(sentence_token, dtype=torch.long).unsqueeze(0).to(device)

y = model.generate(x, 50)

print(tokenizer.decode(y[0].tolist()))

ユーザー：日本の首都はどこですか？
アシスタント: 1977
アシスタント：大都市で2017

オーストリアの首都の高い首都は、ブルジュ・コードにある
